# SKILL智能体开发模式

## 1. 提示词的结构 (Skill)

In [ ]:
# 先定义skill数据类
from typing import TypedDict

class Skill(TypedDict):
    name: str            # Skill的名字
    description: str     # Skill的作用描述 (问题由模型识别，匹配description描述，加载content)
    content: str         # Skill具体的提示词

- 匹配规则
    - 先匹配name与description。匹配上加载content
    - 渐进式提示词披露，按需加载

## 2. 编写提示词 (Skill)

In [ ]:
SKILLS: list[Skil l] = [
    {
        "name": "sales_analytics",
        "description": "用于销售数据分析的数据库模式和业务逻辑，包括客户、订单和收入。",
        "content": """# 销售分析模式

## 数据表

### customers (客户表)
- customer_id (主键)
- name (姓名)
- email (邮箱)
- signup_date (注册日期)
- status (状态: active活跃/inactive非活跃)
- customer_tier (客户等级: bronze铜/silver银/gold金/platinum铂金)

### orders (订单表)
- order_id (主键)
- customer_id (外键 -> customers)
- order_date (订单日期)
- status (状态: pending待处理/completed已完成/cancelled已取消/refunded已退款)
- total_amount (总金额)
- sales_region (销售区域: north北/south南/east东/west西)

### order_items (订单明细表)
- item_id (主键)
- order_id (外键 -> orders)
- product_id (产品ID)
- quantity (数量)
- unit_price (单价)
- discount_percent (折扣百分比)

## 业务逻辑

**活跃客户**: status = 'active' AND signup_date <= CURRENT_DATE - INTERVAL '90 days'

**收入计算**: 仅统计状态为 'completed' 的订单。使用 orders 表中的 total_amount 字段，该字段已计算折扣。

**客户生命周期价值 (CLV)**: 客户所有已完成订单金额的总和。

**高价值订单**: total_amount > 1000 的订单。

## 示例查询

-- 获取上季度收入排名前10的客户
SELECT
    c.customer_id,
    c.name,
    c.customer_tier,
    SUM(o.total_amount) as total_revenue
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
WHERE o.status = 'completed'
  AND o.order_date >= CURRENT_DATE - INTERVAL '3 months'
GROUP BY c.customer_id, c.name, c.customer_tier
ORDER BY total_revenue DESC
LIMIT 10;
""",
    },
    {
        "name": "inventory_management",
        "description": "用于库存跟踪的数据库模式和业务逻辑，包括产品、仓库和库存水平。",
        "content": """# 库存管理模式

## 数据表

### products (产品表)
- product_id (主键)
- product_name (产品名称)
- sku (库存单位)
- category (类别)
- unit_cost (单位成本)
- reorder_point (再订货点：需要补货的最低库存水平)
- discontinued (是否停产)

### warehouses (仓库表)
- warehouse_id (主键)
- warehouse_name (仓库名称)
- location (位置)
- capacity (容量)

### inventory (库存表)
- inventory_id (主键)
- product_id (外键 -> products)
- warehouse_id (外键 -> warehouses)
- quantity_on_hand (现有库存量)
- last_updated (最后更新时间)

### stock_movements (库存移动记录表)
- movement_id (主键)
- product_id (外键 -> products)
- warehouse_id (外键 -> warehouses)
- movement_type (移动类型：inbound入库/outbound出库/transfer调拨/adjustment调整)
- quantity (数量：入库为正，出库为负)
- movement_date (移动日期)
- reference_number (参考编号)

## 业务逻辑

**可用库存**: 来自 inventory 表中 quantity_on_hand > 0 的记录

**需要补货的产品**: 所有仓库的 quantity_on_hand 总和小于或等于产品 reorder_point 的产品

**仅限活跃产品**: 排除 discontinued = true 的产品，除非专门分析已停产项目

**库存估值**: 每个产品的 quantity_on_hand * unit_cost

## 示例查询

-- 查找所有仓库中库存低于再订货点的产品
SELECT
    p.product_id,
    p.product_name,
    p.reorder_point,
    SUM(i.quantity_on_hand) as total_stock,
    p.unit_cost,
    (p.reorder_point - SUM(i.quantity_on_hand)) as units_to_reorder
FROM products p
JOIN inventory i ON p.product_id = i.product_id
WHERE p.discontinued = false
GROUP BY p.product_id, p.product_name, p.reorder_point, p.unit_cost
HAVING SUM(i.quantity_on_hand) <= p.reorder_point
ORDER BY units_to_reorder DESC;
""",
    },
]

## 3. 加载提示词 (Skill)的tool

In [ ]:
from langchain.tools import tool
@tool
def load_skill(skill_name: str) -> str:
    """
    将技能的完整内容加载到Agent的上下文中。

    当您需要关于如何处理特定类型请求的详细信息时，应使用此功能。这将为您提供该技能领域的全面说明、政策和指南。

    Args：
        skill_name: 要加载的技能名称（例如，“费用报销”，“旅行预订”,"销售分析"，"仓库管理"）
    """
    for skill in SKILLS:
        if skill["name"] == skill_name:
            return F"已加载技能：{skill_name}\n\n{skill['content']}"
    # 不存在技能
    avaible = ",".joins(s["name"]  for s in  SKILLS)
    return F"未找到技能，可用的技能有：{avaible}"

- Skill的存储形式可以是: 文件、CSV文件、MDwenjian、HTML/XML文件
- 可以是MongDB、Redis、NoSQL
- PostgreSQL、SQLite、Oracle
- 知识图谱(Obsidian)

## 4. 使用Middleware的隐式挂载tool

- 在模型调用之前，加强提示词的上下文
    - 中间件的wrap_model_call: 按需披露

所谓隐式

In [ ]:
from langchain.agents.middleware import ModelRequest, ModelResponse, AgentMiddleware
from langchain.messages import SystemMessage
from typing import Callable

class SkillMiddleware(AgentMiddleware):
    # 覆盖tool数据成员
    tools = [load_skill]

    def __init__(self):
        #根据SKILL列表构建提示词（Skill添加到提示词的长下文中）
        skills_list  = []
        for skill in SKILLS:
            skills_list.append(
                f"- **{skill['name']}**: {skill['description']}"    #  只添加skill的name与description，不添加content（渐进式披露）
            )
        self.skills_prompt = "\n".join(skills_list)

    # 改变系统提示词
    def wrap_model_call(self, request: ModelRequest, handler:Callable[[ModelRequest], ModelResponse]):
        """
        同步：技能藐视注入到系统提示词。
        """
        # 动态注入提示词
        skill_addendum = (F"\n\n ## 可用技能\n\n{self.skills_prompt}\n\n"
                          "当你需要处理处理特定类型请求的详细信息时，"
                          "请使用load_skill工具")
        # 把上面的第一级披露添加到系统提示词（二级披露，使用tool）
        # new_content = list(request.system_message.content_blocks) + [   # 没有使用message的content或者{"role":"user"}
        #     {
        #         "type": "text",
        #         "text": skill_addendum    # 主要方便支持多模态
        #     }
        # ]
        new_content = request.system_message.content + skill_addendum  # 只支持LLM（非多模态）
        new_system_message = SystemMessage(content=new_content)
        new_system_message.pretty_print()
        # 更新request
        modified_request = request.override(system_message=new_system_message)
        return handler(modified_request)

## 5. 创建智能体